# Develop the Stance Prompt on Thirty Turns

| Item | Description |
|---|---|
| Goal | Try the stance-extraction prompt on thirty legislator turns and read every output next to its source sentences, before any production run. A record is a position a member takes: the sentence numbers that carry it, the policy object as a neutral noun phrase, and `support` or `concern`. The model picks sentence numbers, never text, so evidence cannot be invented. |
| Input | `member_turns.parquet` from T2.2.1; the model Qwen/Qwen3.5-4B from the project cache. |
| Output | Call logs in the T2.2 Interim folder, `dev/status/<packageId>_<turn>.jsonl`, one file per turn. Nothing else is saved; development output is for reading. |
| Variables | • `records`: the parsed records of a turn, each with `sentences`, `topic`, `stance`, and `topic_lint`<br>• `errors`: named parse errors per turn<br>• `n_merged`: records merged because they repeated one position on adjacent sentences |
| Run | Last run 2026-08-22 on one A40: 30 turns, 39 calls, 86 seconds, the third wording. The outputs shown are from that run. Not rerun: this wording is the one the production run in T2.2.3 used, and the stored logs are the record of its development. |


## Step 1. Import packages


In [ ]:
import hashlib
import json
import os
import re
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


## Step 2. Define file locations


In [ ]:
PROJECT_FOLDER = Path.cwd().resolve().parents[2]  # <pipeline>/<section>/<subfolder>/ sits three levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T - Three Step Text Analysis Pipeline/T2 - Topic Extraction and Stance Analysis/T2.2 - Track 2 - Legislator Stance Extraction")
NOTEBOOK_NAME = "T2.2.2 - Draft - Develop the Stance Prompt on Thirty Turns"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
DEV_FOLDER = INTERIM_FOLDER / "dev"
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T - Three Step Text Analysis Pipeline/T2 - Topic Extraction and Stance Analysis" / ".env"
for folder in (INTERIM_FOLDER, PROCESSED_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(MODEL_CACHE)
# A token is needed only for gated models. Qwen3.5-4B and bge-large-en-v1.5 are open; the lines stay so the cell also serves a gated model.
if ENV_FILE.exists():
    for line in ENV_FILE.read_text().splitlines():
        name, _, value = line.strip().partition("=")
        if name == "HF_TOKEN" and value:
            os.environ["HF_TOKEN"] = value.strip().strip('"').strip("'")


## Step 3. Define the prompt

The prompt names the three fields of a record and the two stances. It asks the model to check that each record reads as "the member supports [topic]" or "the member is concerned about [topic]". When no sentence takes a position, the model returns an empty list. Procedure, courtesies, and questions without a position are excluded on purpose. The lint flags directional words inside a topic phrase for review.


In [ ]:
PROMPT_VERSION = "stance-extraction-0.3-dev"  # stamped on every logged call; the production run kept this recorded name
WINDOW_SENTENCES = 40  # a longer turn is sent in consecutive windows of this size, with turn-global sentence numbers
STANCES = ("support", "concern")

SYSTEM = ("You are an expert reader of United States congressional hearing transcripts. "
          "You report only what the text states.")

USER = """Below is one speaking turn by a member of Congress, split into numbered sentences.

Task: find every sentence in which the member takes a directional position on a policy object, and return one record per position.

A record has three fields:
- "sentences": the sentence number(s) expressing the position (1-3 consecutive numbers);
- "topic": the policy object, as a short and specific neutral noun phrase (the policy, program, regulation, technology, practice, or budget item itself; never directional wording; name the specific object, not the agency or field around it);
- "stance": "support" if the member endorses, advocates, or urges adoption, funding, or expansion of the topic; "concern" if the member expresses worry, criticism, or objection toward the topic, or urges its restriction, delay, or scrutiny.

The stance is toward the topic exactly as you phrase it. Before returning a record, check that it reads sensibly as "the member supports [topic]" or "the member is concerned about [topic]". If the topic is a problem or harm, the stance toward it is "concern", never "support".

The topic must be the object the member takes a position ON. Entities mentioned only as examples, comparisons, or asides are not topics, and a bare number or amount is never a topic. If several consecutive sentences express the same position on the same topic, return one record covering them, not one record per sentence.

Example: "Youth vaping is an epidemic we cannot ignore." -> {{"sentences": [n], "topic": "youth e-cigarette use", "stance": "concern"}}
Example: "This bill funds the research our labs badly need." -> {{"sentences": [n], "topic": "research funding bill", "stance": "support"}}

Do not return records for: procedure (recognitions, time, the record), courtesies, introductions, logistics, questions asked without an expressed position, or factual statements without evaluation. If no sentence takes a directional position, return an empty list. When in doubt, do not extract.

--- TURN (Speaker: {speaker}) ---
{numbered}
--- END OF TURN ---

Return only a JSON list of records. Your very first output character must be [."""

# Directional words that must not appear inside a topic phrase; direction belongs in the stance field.
# The lint flags a phrase for review. It does not rewrite it.
DIRECTIONAL_LINT = re.compile(
    r"\b(?:ban(?:ning)?|block(?:ing)?|stop(?:ping)?|halt(?:ing)?|"
    r"oppos(?:e|ing|ition)|support(?:ing)?|defend(?:ing)?|protect(?:ing|ion)?|"
    r"against|anti|pro|promot(?:e|ing|ion)|expand(?:ing)?|restrict(?:ing|ion)?|"
    r"repeal(?:ing)?|eliminat(?:e|ing|ion))\b",
    re.IGNORECASE,
)


def lint_topic(phrase):
    """The directional word a topic phrase contains, or None if clean."""
    match = DIRECTIONAL_LINT.search(phrase or "")
    return match.group(0) if match else None


## Step 4. Define the windows and the parser

A turn longer than 40 sentences is sent in consecutive windows. The sentence numbers continue across windows, so a record means the same thing whichever window produced it. The parser checks every property code can check and names each failure. Records that repeat one position on adjacent sentences are merged, so a verbose member does not count more.


In [ ]:
def render_user(sentences, speaker, start=1):
    """The extraction prompt for one turn, or one window of a long turn."""
    return USER.format(speaker=speaker, numbered=format_numbered(sentences, start=start))


def windows(sentences, size=WINDOW_SENTENCES):
    """Yield (first sentence number, sentences) for each window of a turn; a short turn is one window."""
    for offset in range(0, max(len(sentences), 1), size):
        yield offset + 1, sentences[offset:offset + size]


def parse_records(output, n_sentences, start=1):
    """Validate one model output into records and a list of named errors.

    Every check that code can make is made here: JSON shape, field presence, stance vocabulary,
    sentence indices within the window (at most 3, consecutive), and the directional lint on the topic.
    A record that fails a structural check is dropped and counted; a lint hit keeps the record with a flag.
    """
    errors = []
    text = output.strip()
    if text.startswith("```"):  # a markdown fence is tolerated but counted
        errors.append("markdown_fence")
        text = re.sub(r"^```[a-zA-Z]*\s*|\s*```$", "", text).strip()
    try:
        raw = json.loads(text)
    except json.JSONDecodeError:
        return [], errors + ["json_parse"]
    if not isinstance(raw, list):
        return [], errors + ["not_a_list"]
    records = []
    for item in raw:
        if not isinstance(item, dict):
            errors.append("record_not_object")
            continue
        topic, stance, sent = item.get("topic"), item.get("stance"), item.get("sentences")
        if isinstance(sent, int):
            sent = [sent]
        if not (isinstance(topic, str) and topic.strip()):
            errors.append("missing_topic")
            continue
        if stance not in STANCES:
            errors.append("bad_stance")
            continue
        if not (isinstance(sent, list) and sent and all(isinstance(i, int) for i in sent)):
            errors.append("bad_sentences_field")
            continue
        sent = sorted(set(sent))
        if sent[0] < start or sent[-1] > start + n_sentences - 1:
            errors.append("index_out_of_range")
            continue
        if len(sent) > 3 or sent[-1] - sent[0] != len(sent) - 1:
            errors.append("indices_not_consecutive")
            continue
        records.append({"sentences": sent, "topic": topic.strip(), "stance": stance, "topic_lint": lint_topic(topic)})
    return records, errors


def merge_adjacent_records(records):
    """Merge records with the same topic and stance whose sentences touch or sit within two sentences; returns the list and the merge count."""
    if not records:
        return records, 0
    ordered = sorted(records, key=lambda r: (r["topic"].strip().lower(), r["stance"], r["sentences"][0]))
    merged, n_merged = [], 0
    for record in ordered:
        if merged:
            prev = merged[-1]
            if (prev["topic"].strip().lower() == record["topic"].strip().lower() and prev["stance"] == record["stance"]
                    and record["sentences"][0] - prev["sentences"][-1] <= 2):
                prev["sentences"] = sorted(set(prev["sentences"] + record["sentences"]))
                n_merged += 1
                continue
        merged.append(dict(record))
    merged.sort(key=lambda r: r["sentences"][0])
    return merged, n_merged


## Step 5. Define the model call and the log

One turn per call, fresh context, near-greedy sampling, 1,024 new tokens at most. Every call is logged with its prompt version, so each development round stays distinguishable on disk.


In [ ]:
MAX_NEW_TOKENS = 1024  # a long window can carry many records; generation stops at the end marker, so the cap costs nothing on short turns
N_DEV = 30

GENERATION = {"do_sample": True, "temperature": 0.1}  # near-greedy sampling, the same setting as T1


def render(tokenizer, system, user):
    """Apply the model's chat template, with the thinking mode off where the template offers it."""
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate(model, tokenizer, system, user, max_new_tokens):
    """One prompt, one completion, fresh context. Also returns whether the completion ran into the token cap."""
    prompt = render(tokenizer, system, user)
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    n_in = ids["input_ids"].shape[-1]
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id, **GENERATION)
    generated = out[0][n_in:]
    hit_cap = len(generated) >= max_new_tokens
    return tokenizer.decode(generated, skip_special_tokens=True).strip(), hit_cap


class CallLog:
    """One JSON line per model call, written as it happens: stage, level, position, a hash of the exact input, token counts, time, and the full output. The same format as T1."""

    def __init__(self, path, document_id, model_name, prompt_version):
        self.path = Path(path) if path else None
        self.document_id = document_id
        self.model_name = model_name
        self.prompt_version = prompt_version
        self.records = []
        if self.path:
            self.path.parent.mkdir(parents=True, exist_ok=True)

    def write(self, stage, level, index, n_inputs, prompt_text, output, tokenizer, seconds):
        record = {
            "document_id": self.document_id, "model": self.model_name, "prompt_version": self.prompt_version,
            "stage": stage, "level": level, "index": index, "n_inputs": n_inputs,
            "input_sha256": hashlib.sha256(prompt_text.encode("utf-8")).hexdigest(),
            "input_chars": len(prompt_text),
            "input_tokens": len(tokenizer.encode(prompt_text, add_special_tokens=False)),
            "output_tokens": len(tokenizer.encode(output, add_special_tokens=False)),
            "output_chars": len(output), "seconds": round(seconds, 2),
            "at": datetime.now(timezone.utc).isoformat(timespec="seconds"), "output": output,
        }
        self.records.append(record)
        if self.path:
            with self.path.open("a") as log_file:
                log_file.write(json.dumps(record) + "\n")
                log_file.flush()


## Step 6. Set the run


In [1]:
MODEL = os.environ.get("MODEL", "Qwen/Qwen3.5-4B")
print(f"model    : {MODEL}")
print(f"prompt   : {PROMPT_VERSION}")
print(f"window   : {WINDOW_SENTENCES} sentences")
print(f"sampling : {GENERATION}")
print(f"gpu      : {torch.cuda.get_device_name(0)}")


model    : Qwen/Qwen3.5-4B
prompt   : stance-extraction-0.3-dev
window   : 40 sentences
sampling : {'do_sample': True, 'temperature': 0.1}
gpu      : NVIDIA A40


## Step 7. Print the prompt in full


In [2]:
print("=" * 78)
print(f"SYSTEM   ({PROMPT_VERSION})")
print("=" * 78)
print(SYSTEM)
print()
print("=" * 78)
print("USER")
print("=" * 78)
print(USER)


SYSTEM   (stance-extraction-0.3-dev)
You are an expert reader of United States congressional hearing transcripts. You report only what the text states.

USER
Below is one speaking turn by a member of Congress, split into numbered sentences.

Task: find every sentence in which the member takes a directional position on a policy object, and return one record per position.

A record has three fields:
- "sentences": the sentence number(s) expressing the position (1-3 consecutive numbers);
- "topic": the policy object, as a short and specific neutral noun phrase (the policy, program, regulation, technology, practice, or budget item itself; never directional wording; name the specific object, not the agency or field around it);
- "stance": "support" if the member endorses, advocates, or urges adoption, funding, or expansion of the topic; "concern" if the member expresses worry, criticism, or objection toward the topic, or urges its restriction, delay, or scrutiny.

The stance is toward the t

## Step 8. Draw the development sample

Thirty turns, spread over chamber and turn length. Long opening statements test whether records are missed. Short procedural turns test whether records are invented; the right answer for most of them is an empty list. The draw is seeded, so every round reads the same thirty turns.


In [3]:
turns = pd.read_parquet(PROCESSED_FOLDER / "member_turns.parquet")
turns = turns.loc[turns["n_sentences"] > 0]

turns["bucket"] = pd.cut(turns["n_sentences"], [0, 3, 12, 40, 10_000], labels=["short", "medium", "long", "very_long"])
per_cell = max(1, N_DEV // (2 * 4))
dev = (turns.groupby(["chamber", "bucket"], observed=True, group_keys=False)
       .apply(lambda g: g.sample(min(per_cell, len(g)), random_state=1089))
       .head(N_DEV).reset_index(drop=True))

print(dev.groupby(["chamber", "bucket"], observed=True).size().to_string())
print(f"\n{len(dev)} dev turns, {dev['n_sentences'].sum():,} sentences")


chamber  bucket   
HOUSE    short        3
         medium       3
         long         3
         very_long    3
JOINT    short        3
         medium       3
         long         3
         very_long    2
SENATE   short        3
         medium       3
         long         1

30 dev turns, 591 sentences


## Step 9. Load the model


In [4]:
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype="auto", device_map="cuda")
model.eval()
print(f"loaded in {time.time() - t0:.0f}s, {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B parameters")


loaded in 5s — 4.21B params


## Step 10. Extract from the thirty turns

Each turn seeds the sampler from a hash of its key, so its result does not depend on the order of the run.


In [5]:
results = []
started = time.time()
n_calls = 0
for row in dev.itertuples():
    key = f"{row.packageId}:{row.turn_idx}"
    torch.manual_seed(int.from_bytes(hashlib.sha256(key.encode()).digest()[:4], "big"))
    speaker = f"{row.honorific or ''} {row.surname or ''}".strip() or "Unidentified"
    log = CallLog(DEV_FOLDER / "status" / f"{row.packageId}_{row.turn_idx}.jsonl", key, MODEL, prompt_version=PROMPT_VERSION)

    records, errors = [], []
    for w_index, (w_start, w_sents) in enumerate(windows(list(row.sentences))):
        user = render_user(w_sents, speaker, start=w_start)
        t0 = time.time()
        output, hit_cap = generate(model, tokenizer, SYSTEM, user, max_new_tokens=MAX_NEW_TOKENS)
        log.write("extract", 0, w_index, 1, user, output, tokenizer, time.time() - t0)
        n_calls += 1
        w_records, w_errors = parse_records(output, len(w_sents), start=w_start)
        if hit_cap:
            w_errors.append("hit_token_cap")
        records.extend(w_records)
        errors.extend(w_errors)
    records, n_merged = merge_adjacent_records(records)
    results.append({"key": key, "speaker": speaker, "sentences": row.sentences, "records": records, "errors": errors, "n_merged": n_merged})

total_merged = sum(r["n_merged"] for r in results)
print(f"{len(results)} turns / {n_calls} calls in {time.time() - started:.0f}s ({(time.time() - started) / n_calls:.1f}s per call); {total_merged} adjacent duplicates merged")


30 turns / 39 calls in 86s (2.2s per call); 1 adjacent duplicates merged


## Step 11. Read the output

Every turn is printed with its numbered sentences and the records drawn from them. Things to look for: questions without a position wrongly extracted, directional topic phrases, and missed positions phrased as concern. Also empty lists on turns that plainly take a position.


In [6]:
for r in results:
    print("=" * 78)
    print(f"{r['key']}  [{r['speaker']}]  {len(r['records'])} record(s)" + (f"  ERRORS: {r['errors']}" if r["errors"] else ""))
    print(format_numbered(list(r["sentences"])[:12]))
    if len(r["sentences"]) > 12:
        print(f"  ... {len(r['sentences']) - 12} more sentences")
    for rec in r["records"]:
        lint = f"  [LINT: {rec['topic_lint']}]" if rec["topic_lint"] else ""
        print(f"  -> {rec['sentences']}  ({rec['stance']:<7})  {rec['topic']}{lint}")
    if not r["records"] and not r["errors"]:
        print("  -> (no directional position: empty list)")
    print()


CHRG-109hhrg20007:162  [Mr. Udall]  0 record(s)
[1] Dr. Benzakein.
  -> (no directional position — empty list)

CHRG-114hhrg97769:74  [Ms. Esty]  0 record(s)
[1] And if you can briefly talk a little bit about, for example, the Home Energy Solutions program, how that works in Connecticut again to put the power in the hands of consumers and businesses to make those choices.
  -> (no directional position — empty list)

CHRG-111hhrg57174:188  [Mr. Garamendi]  0 record(s)
[1] And you don't think you will have to pay for that?
  -> (no directional position — empty list)

CHRG-115hhrg28414:167  [Mr. Dunn]  0 record(s)
[1] Thank you.
[2] So Mr. Gerstenmaier, in the cost end of the training side, you now have three new, different manned capsules.
[3] And I presume your astronauts have to train on each of these.
[4] But going forward, how are you going to segment that training?
[5] Is it everybody trains on everything or 1/3 here, 1/3 there or how does that work?
  -> (no directional position — 

## Step 12. Summarize the checks

These rates become the alarm bands of the production shards in T2.2.3. On thirty turns they only point a direction.


In [7]:
n_records = sum(len(r["records"]) for r in results)
error_counts = Counter(e for r in results for e in r["errors"])
stances = Counter(rec["stance"] for r in results for rec in r["records"])
lints = [rec["topic"] for r in results for rec in r["records"] if rec["topic_lint"]]

clean = sum(1 for r in results if not r["errors"])
print(f"turns parsed clean : {clean}/{len(results)}")
print(f"records extracted  : {n_records} ({n_records / len(results):.1f} per turn)")
print(f"empty-list turns   : {sum(1 for r in results if not r['records'])}")
print(f"stance mix         : {dict(stances)}")
print(f"errors             : {dict(error_counts) or 'none'}")
print(f"topic lint hits    : {len(lints)}")
for topic in lints[:5]:
    print(f"   {topic!r}")


turns parsed clean : 29/30
records extracted  : 38 (1.3 per turn)
empty-list turns   : 24
stance mix         : {'support': 12, 'concern': 26}
errors             : {'indices_not_consecutive': 1}
topic lint hits    : 2
   'NASA Inspector General Robert Cobb blocking notification to State Department'
   'NASA Inspector General Robert Cobb blocking NASA OIG investigation'
